# 02 — Building a zoning system and centroid connectors

Travel demand models divide space into **traffic analysis zones (TAZs)**. Each zone
gets a **centroid** — an abstract node where all trips start/end — connected to the
real network by **centroid connectors**.

Here we build a hexagonal zoning system for Nauru (the smallest example model),
exactly like the AequilibraE documentation's *Create zoning system* example:

1. compute the network's convex hull and extent;
2. tile it with hexagons sized so we get roughly the number of zones we want;
3. keep the hexagons that actually touch the network;
4. add centroids and connect them to the road network per mode.


In [1]:
from math import sqrt
from pathlib import Path
from tempfile import gettempdir
from uuid import uuid4

import shapely.wkb
from shapely.geometry import Point

from aequilibrae.utils.create_example import create_example
from aequilibrae.utils.db_utils import read_and_close

fldr = str(Path(gettempdir()) / uuid4().hex)
project = create_example(fldr, "nauru")

In [2]:
zones_wanted = 10

network = project.network
geo = network.convex_hull()          # convex hull of all link geometry
extent = network.extent()            # bounding box polygon

# Hexagon side length that yields ~`zones_wanted` hexes over the hull area
zone_area = geo.area / zones_wanted
zone_side = sqrt(2 * sqrt(3) * zone_area / 9)
print(f"hull area {geo.area:.6f} deg^2 -> hexagon side {zone_side:.5f} deg")

hull area 0.001782 deg^2 -> hexagon side 0.00828 deg


The hexagonal tiling itself is done **in SQL** with the SpatiaLite `HexagonalGrid`
function (served by AequilibraE's built-in spatial engine — no native extension needed).


In [3]:
b = extent.bounds
with read_and_close(project.path_to_file, spatial=True) as conn:
    sql = "select st_asbinary(HexagonalGrid(GeomFromWKB(?), ?, 0, GeomFromWKB(?)))"
    grid = conn.execute(sql, [extent.wkb, zone_side, Point(b[2], b[3]).wkb]).fetchone()[0]

grid = [p for p in shapely.wkb.loads(grid).geoms if p.intersects(geo)]
print(f"{len(grid)} hexagons intersect the network hull")

18 hexagons intersect the network hull


In [4]:
# Register each hexagon as a zone, with a centroid at its centre of mass
zoning = project.zoning
for zone_id, hexagon in enumerate(grid, 1):
    zone = zoning.new(zone_id)
    zone.geometry = hexagon
    zone.save()
    # place the centroid (robust=True nudges it if it would collide with an existing node)
    zone.add_centroid(None)

project.network.count_centroids()

0

## Centroid connectors

`connect_mode` links each centroid to nearby network nodes reachable by a given mode.
The number of connectors per centroid is a key modeling decision — too few creates
artificial bottlenecks, too many lets traffic bypass the real network.


In [5]:
for zone_id in zoning.all_zones():
    zone = zoning.get(zone_id)
    zone.connect_mode(mode_id="c", connectors=1)

C:\Users\Riz\Desktop\AequilibraE\venv\Lib\site-packages\geopandas\array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


C:\Users\Riz\Desktop\AequilibraE\venv\Lib\site-packages\geopandas\array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


C:\Users\Riz\Desktop\AequilibraE\venv\Lib\site-packages\geopandas\array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


C:\Users\Riz\Desktop\AequilibraE\venv\Lib\site-packages\geopandas\array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


C:\Users\Riz\Desktop\AequilibraE\venv\Lib\site-packages\geopandas\array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


C:\Users\Riz\Desktop\AequilibraE\venv\Lib\site-packages\geopandas\array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


C:\Users\Riz\Desktop\AequilibraE\venv\Lib\site-packages\geopandas\array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


C:\Users\Riz\Desktop\AequilibraE\venv\Lib\site-packages\geopandas\array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


C:\Users\Riz\Desktop\AequilibraE\venv\Lib\site-packages\geopandas\array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


C:\Users\Riz\Desktop\AequilibraE\venv\Lib\site-packages\geopandas\array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


In [6]:
# The connectors are ordinary links with link_type 'centroid_connector'
links = project.network.links.data
connectors = links[links.link_type == "centroid_connector"]
print(f"{len(connectors)} centroid connectors created")

10 centroid connectors created


In [7]:
# JupyterGIS map helper ------------------------------------------------------
# GISDocument is JupyterGIS' notebook API: it builds a live, QGIS-like map
# document rendered directly in JupyterLab. Layers added from GeoDataFrames
# are converted to GeoJSON on the fly.
import json
from jupytergis import GISDocument

OSM_TILES = "https://tile.openstreetmap.org/{z}/{x}/{y}.png"

def new_map(gdf_for_extent=None, zoom=12):
    """Create a GISDocument centred on a layer, with an OpenStreetMap basemap."""
    kwargs = {}
    if gdf_for_extent is not None:
        b = gdf_for_extent.total_bounds  # (minx, miny, maxx, maxy)
        kwargs = {"longitude": (b[0] + b[2]) / 2, "latitude": (b[1] + b[3]) / 2, "zoom": zoom}
    doc = GISDocument(**kwargs)
    doc.add_raster_layer(OSM_TILES, name="OpenStreetMap", attribution="(C) OpenStreetMap contributors", opacity=0.6)
    return doc

def add_gdf(doc, gdf, name, **kwargs):
    """Add a GeoDataFrame to the map as a GeoJSON layer."""
    return doc.add_geojson_layer(data=json.loads(gdf.to_json()), name=name, **kwargs)

In [8]:
from jupytergis_lab.notebook.symbology import constant

zones_gdf = project.zoning.data
nodes = project.network.nodes.data

doc = new_map(zones_gdf, zoom=13)
add_gdf(doc, zones_gdf, "zones", opacity=0.3, symbology=[[constant("#10b981").encoding("fill")]])
add_gdf(doc, links[links.link_type != "centroid_connector"], "roads",
        symbology=[[constant("#334155").encoding("stroke")]])
add_gdf(doc, connectors, "connectors", symbology=[[constant("#dc2626").encoding("stroke")]])
add_gdf(doc, nodes[nodes.is_centroid == 1], "centroids", symbology=[[constant("#7c3aed").encoding("fill")]])
doc

C:\Users\Riz\AppData\Local\Temp\ipykernel_23280\1823397229.py:16: UserWarning: The JupyterGIS Python API is better experienced in the xeus-python kernel which supports awaiting comm messages
  doc = GISDocument(**kwargs)


In [9]:
project.close()

---
**Next:** [03 — Path computation and skimming](03_paths_and_skimming.ipynb): building graphs
and measuring zone-to-zone costs.
